In [1]:
import numpy as np
import pandas as pd

In [2]:
import os
print(os.path.exists('../data/processed/train_optimized.pkl'))

True


In [3]:
train = pd.read_pickle('../data/processed/train_optimized.pkl')
print(train.shape)
print(f'{train.memory_usage(deep=True).sum():,}')

(590540, 434)
953,877,651


In [4]:
train = train.copy()

In [5]:
train['trans_day'] = (train['TransactionDT'] // 86400).astype(np.int16)
train['trans_hour'] = ((train['TransactionDT'] % 86400) // 3600).astype(np.int8)
train['trans_weekday'] = (train['trans_day'] % 7).astype(np.int8)

In [6]:
train = train.sort_values('TransactionDT').reset_index(drop=True)
train['uid'] = train['card1'].astype(str) + '_' + train['addr1'].astype(str)
print(train['uid'].nunique())
print(train[['TransactionDT', 'card1', 'addr1', 'uid']].head(10))

37531
   TransactionDT  card1  addr1          uid
0          86400  13926  315.0  13926_315.0
1          86401   2755  325.0   2755_325.0
2          86469   4663  330.0   4663_330.0
3          86499  18132  476.0  18132_476.0
4          86506   4497  420.0   4497_420.0
5          86510   5937  272.0   5937_272.0
6          86522  12308  126.0  12308_126.0
7          86529  12695  325.0  12695_325.0
8          86535   2803  337.0   2803_337.0
9          86536  17399  204.0  17399_204.0


In [7]:
uid_grp = train.groupby('uid')['TransactionAmt']
train['uid_transaction_count'] = uid_grp.cumcount()
train['uid_mean_amount'] = uid_grp.transform(lambda x: x.shift(1).expanding().mean())
train['uid_std_amount'] = uid_grp.transform(lambda x: x.shift(1).expanding().std())
train['uid_amount_ratio'] = train['TransactionAmt']/train['uid_mean_amount']
train['uid_amount_zscore'] = (train['TransactionAmt'] - train['uid_mean_amount']) / (train['uid_std_amount'])
train['uid_mean_hour'] = train.groupby('uid')['trans_hour'].transform(lambda x: x.shift(1).expanding().mean())
print(train[['uid', 'TransactionAmt', 'uid_transaction_count', 'uid_mean_amount', 'uid_std_amount', 'uid_amount_ratio', 'uid_amount_zscore']].head(20))

            uid  TransactionAmt  uid_transaction_count  uid_mean_amount  \
0   13926_315.0       68.500000                    0.0              NaN   
1    2755_325.0       29.000000                    0.0              NaN   
2    4663_330.0       59.000000                    0.0              NaN   
3   18132_476.0       50.000000                    0.0              NaN   
4    4497_420.0       50.000000                    0.0              NaN   
5    5937_272.0       49.000000                    0.0              NaN   
6   12308_126.0      159.000000                    0.0              NaN   
7   12695_325.0      422.500000                    0.0              NaN   
8    2803_337.0       15.000000                    0.0              NaN   
9   17399_204.0      117.000000                    0.0              NaN   
10          NaN       75.887001                    NaN              NaN   
11          NaN       16.495001                    NaN              NaN   
12   3786_204.0       50.

In [8]:
uid_counts = train['uid'].value_counts()
print(uid_counts.head(10))

top_uid = uid_counts.index[0]
print(train[train['uid'] == top_uid][['uid', 'TransactionDT', 'TransactionAmt',
    'uid_transaction_count', 'uid_mean_amount',
    'uid_amount_ratio', 'uid_amount_zscore']].head(15))

uid
17188_299.0    5885
12695_325.0    5774
9500_204.0     4665
12839_264.0    3547
16132_299.0    3537
15497_299.0    3426
9500_272.0     2725
7664_264.0     2552
7207_204.0     2327
9112_441.0     2290
Name: count, dtype: int64
             uid  TransactionDT  TransactionAmt  uid_transaction_count  \
65   17188_299.0          87650      114.949997                    0.0   
81   17188_299.0          87868      104.949997                    1.0   
89   17188_299.0          88042      318.950012                    2.0   
159  17188_299.0          89072       92.949997                    3.0   
163  17188_299.0          89103      973.950012                    4.0   
238  17188_299.0          90163      226.000000                    5.0   
293  17188_299.0          91119      117.000000                    6.0   
394  17188_299.0          92760      344.950012                    7.0   
409  17188_299.0          93055      210.949997                    8.0   
421  17188_299.0          9335

In [9]:
print(f"NaN uids: {train['uid'].isna().sum()}")
train['uid'] = train['uid'].fillna('unknown_addr')
print(f"NaN uids after fix: {train['uid'].isna().sum()}")

NaN uids: 65706
NaN uids after fix: 0


In [10]:
train['uid_mean_amount'] = train['uid_mean_amount'].fillna(-1)
train['uid_std_amount'] = train['uid_std_amount'].fillna(-1)
train['uid_amount_ratio'] = train['uid_amount_ratio'].fillna(-1)
train['uid_amount_zscore'] = train['uid_amount_zscore'].fillna(-1)
train['uid_mean_hour'] = train['uid_mean_hour'].fillna(-1)
print(train[['uid_mean_amount', 'uid_std_amount', 'uid_amount_ratio',
             'uid_amount_zscore', 'uid_mean_hour']].isnull().sum())

uid_mean_amount      0
uid_std_amount       0
uid_amount_ratio     0
uid_amount_zscore    0
uid_mean_hour        0
dtype: int64


In [11]:
from sklearn.model_selection import train_test_split

X = train.drop(columns=['isFraud', 'TransactionID', 'uid'])
y = train['isFraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(X_train.shape, X_test.shape)

(472432, 441) (118108, 441)


In [12]:
print(X_train.dtypes.value_counts())
print(X_train.select_dtypes(include=['category', 'object', 'str']).columns.tolist())

float32     384
uint16       13
category     12
float64       7
category      4
uint8         2
int8          2
uint32        1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
int16         1
Name: count, dtype: int64
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


In [13]:
import re
from sklearn.preprocessing import LabelEncoder

high_card_cols = ['P_emaildomain', 'R_emaildomain', 'id_30', 'id_31', 'id_33', 'DeviceInfo']
low_card_cols = [col for col in X_train.select_dtypes(include='category').columns
                 if col not in high_card_cols]

# label encode high cardinality
for col in high_card_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    known_classes = set(le.classes_)
    X_test[col] = X_test[col].astype(str).apply(lambda x: x if x in known_classes else 'UNKNOWN')
    le.classes_ = np.append(le.classes_, 'UNKNOWN')
    X_test[col] = le.transform(X_test[col])

# one-hot encode low cardinality
X_train = pd.get_dummies(X_train, columns=low_card_cols)
X_test = pd.get_dummies(X_test, columns=low_card_cols)

# fix column names for LightGBM
X_train.columns = [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in X_train.columns]
X_test.columns = [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in X_test.columns]

print(X_train.shape, X_test.shape)
print(set(X_train.columns) == set(X_test.columns))

(472432, 478) (118108, 478)
True


In [14]:
import lightgbm as lgb

model_lgb_uid = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

model_lgb_uid.fit(X_train, y_train, eval_set=[(X_test, y_test)])

[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 2.383574 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 40832
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 473
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034989 -> initscore=-3.317101
[LightGBM] [Info] Start training from score -3.317101


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,1000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [15]:
from sklearn.metrics import recall_score, precision_score, f1_score

y_proba_uid = model_lgb_uid.predict_proba(X_test)[:, 1]

thresholds = [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]

for thresh in thresholds:
    preds = (y_proba_uid >= thresh).astype(int)
    r = recall_score(y_test, preds)
    p = precision_score(y_test, preds)
    f = f1_score(y_test, preds)
    print(f"thresh={thresh:.2f} | Recall={r:.3f} | Precision={p:.3f} | F1={f:.3f}")

thresh=0.10 | Recall=0.791 | Precision=0.574 | F1=0.666
thresh=0.15 | Recall=0.744 | Precision=0.706 | F1=0.725
thresh=0.20 | Recall=0.705 | Precision=0.784 | F1=0.743
thresh=0.25 | Recall=0.674 | Precision=0.839 | F1=0.748
thresh=0.30 | Recall=0.650 | Precision=0.879 | F1=0.747
thresh=0.40 | Recall=0.589 | Precision=0.922 | F1=0.719
thresh=0.50 | Recall=0.549 | Precision=0.945 | F1=0.694


In [16]:
feat_imp = pd.Series(model_lgb_uid.feature_importances_, index=X_train.columns).sort_values(ascending=False)
uid_features = ['uid_transaction_count', 'uid_mean_amount', 'uid_std_amount','uid_amount_ratio', 'uid_amount_zscore', 'uid_mean_hour']
print(feat_imp[uid_features].sort_values(ascending=False))
print(f'Total features: {len(feat_imp)}')
print(f'UID postion rank position:')
for f in uid_features:
    rank = feat_imp.rank(ascending=False)[f]
    print(f'{f}: {rank}')

uid_transaction_count    786
uid_mean_amount          532
uid_mean_hour            491
uid_std_amount           445
uid_amount_ratio         284
uid_amount_zscore        281
dtype: int32
Total features: 478
UID postion rank position:
uid_transaction_count: 6.0
uid_mean_amount: 8.0
uid_std_amount: 12.0
uid_amount_ratio: 29.0
uid_amount_zscore: 30.0
uid_mean_hour: 10.0


In [17]:
print('TransactionDT' in X_train.columns)
print('uid' in X_train.columns)

True
False


In [18]:
train_uid = X_train['card1'].astype(str) + '_' + X_train['addr1'].astype(str)
test_uid = X_test['card1'].astype(str) + '_' + X_test['addr1'].astype(str)

# step2 build a temp dataframe with uid and time for sorting
train_temp = pd.DataFrame({'uid': train_uid,'TransactionDT': X_train['TransactionDT']}).sort_values('TransactionDT')

test_temp = pd.DataFrame({'uid': test_uid, 'TransactionDT': X_test['TransactionDT']}).sort_values('TransactionDT')

def compute_velocity(temp_df):
    results = []
    grouped = temp_df.groupby('uid')

    for uid, group in grouped:
        times = group['TransactionDT'].values
        idx = group.index.values
        count_1h = []
        count_24h = []

        for i, t in enumerate(times):
            count_1h.append(((times[:i] >= t - 3600) & (times[:i] < t)).sum())
            count_24h.append(((times[:i] >= t - 86400) & (times[:i] < t)).sum())

        results.append(pd.DataFrame({
            'velocity_1h': count_1h,
            'velocity_24h': count_24h
        }, index=idx))

    return pd.concat(results).sort_index()

print("Computing train velocity... (may take 1-2 mins)")
train_velocity = compute_velocity(train_temp)
print("Computing test velocity...")
test_velocity = compute_velocity(test_temp)

X_train['velocity_1h'] = train_velocity['velocity_1h']
X_train['velocity_24h'] = train_velocity['velocity_24h']
X_test['velocity_1h'] = test_velocity['velocity_1h']
X_test['velocity_24h'] = test_velocity['velocity_24h']

print(f"Done. Shape: {X_train.shape}")
print(X_train[['velocity_1h', 'velocity_24h']].describe())

Computing train velocity... (may take 1-2 mins)
Computing test velocity...
Done. Shape: (472432, 480)
         velocity_1h   velocity_24h
count  419709.000000  419709.000000
mean        0.429202       3.530334
std         3.787652      18.416873
min         0.000000       0.000000
25%         0.000000       0.000000
50%         0.000000       1.000000
75%         0.000000       3.000000
max       145.000000     697.000000


In [19]:
model_lgb_vel = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

model_lgb_vel.fit(X_train, y_train, eval_set=[(X_test, y_test)])

[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.395320 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 41184
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 475
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034989 -> initscore=-3.317101
[LightGBM] [Info] Start training from score -3.317101


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,1000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [20]:
y_proba_vel = model_lgb_vel.predict_proba(X_test)[:, 1]

thresholds = [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]

for thresh in thresholds:
    preds = (y_proba_vel >= thresh).astype(int)
    r = recall_score(y_test, preds)
    p = precision_score(y_test, preds)
    f = f1_score(y_test, preds)
    print(f"thresh={thresh:.2f} | Recall={r:.3f} | Precision={p:.3f} | F1={f:.3f}")


thresh=0.10 | Recall=0.786 | Precision=0.572 | F1=0.662
thresh=0.15 | Recall=0.746 | Precision=0.700 | F1=0.722
thresh=0.20 | Recall=0.704 | Precision=0.782 | F1=0.741
thresh=0.25 | Recall=0.668 | Precision=0.838 | F1=0.743
thresh=0.30 | Recall=0.642 | Precision=0.880 | F1=0.742
thresh=0.40 | Recall=0.590 | Precision=0.922 | F1=0.719
thresh=0.50 | Recall=0.541 | Precision=0.943 | F1=0.688


In [21]:
print('D1' in X_train.columns)
print(X_train['D1'].isna().sum())

True
1034


In [22]:
import numpy as np

# Step 1: Build the better uid using card issue date
X_train['D1n'] = np.floor(X_train['TransactionDT'] / (24*60*60)) - X_train['D1']
X_test['D1n'] = np.floor(X_test['TransactionDT'] / (24*60*60)) - X_test['D1']

# Step 2: New precise uid (card1 + addr1 + card issue date)
train_uid2 = X_train['card1'].astype(str) + '_' + X_train['addr1'].astype(str) + '_' + X_train['D1n'].astype(str)
test_uid2 = X_test['card1'].astype(str) + '_' + X_test['addr1'].astype(str) + '_' + X_test['D1n'].astype(str)

# Step 3: Aggregations over the better uid
for col in ['TransactionAmt', 'dist1']:
    X_train[f'uid2_{col}_mean'] = train_uid2.map(X_train.groupby(train_uid2)[col].mean())
    X_train[f'uid2_{col}_std'] = train_uid2.map(X_train.groupby(train_uid2)[col].std())
    X_test[f'uid2_{col}_mean'] = test_uid2.map(X_test.groupby(test_uid2)[col].mean())
    X_test[f'uid2_{col}_std'] = test_uid2.map(X_test.groupby(test_uid2)[col].std())

# Step 4: C13 nunique — tells model if uid contains multiple cards
X_train['uid2_C13_nunique'] = train_uid2.map(X_train.groupby(train_uid2)['C13'].nunique())
X_test['uid2_C13_nunique'] = test_uid2.map(X_test.groupby(test_uid2)['C13'].nunique())

# Drop D1n it's an identifier, not a model feature
X_train.drop('D1n', axis=1, inplace=True)
X_test.drop('D1n', axis=1, inplace=True)

print(X_train.shape)
print(X_train[['uid2_TransactionAmt_mean','uid2_TransactionAmt_std','uid2_C13_nunique']].describe())

(472432, 485)
       uid2_TransactionAmt_mean  uid2_TransactionAmt_std  uid2_C13_nunique
count             418823.000000            314165.000000     418823.000000
mean                 146.489624                71.091034          5.235128
std                  221.920578               145.175977         11.862940
min                    0.588000                 0.000000          1.000000
25%                   56.500000                13.482087          1.000000
50%                   92.397263                34.882659          3.000000
75%                  150.000000                75.507542          6.000000
max                16702.363281             17591.933081        339.000000


In [ ]:
model_lgb_uid2 = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

model_lgb_uid2.fit(X_train, y_train, eval_set=[(X_test, y_test)])

[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.657181 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 42268
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 480
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034989 -> initscore=-3.317101
[LightGBM] [Info] Start training from score -3.317101


In [ ]:
y_proba_vel = model_lgb_uid2.predict_proba(X_test)[:, 1]

thresholds = [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]

for thresh in thresholds:
    preds = (y_proba_vel >= thresh).astype(int)
    r = recall_score(y_test, preds)
    p = precision_score(y_test, preds)
    f = f1_score(y_test, preds)
    print(f"thresh={thresh:.2f} | Recall={r:.3f} | Precision={p:.3f} | F1={f:.3f}")
